In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import cv2
import numpy as np
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor
from sklearn.model_selection import train_test_split

In [ ]:
SEQ_LEN = 16
IMG_SIZE = 112
CLIPS_PER_VIDEO = 3
SEED = 42

DATASET_DIR = "/content/drive/MyDrive/WORK_CCD"

CRASH_DIR = os.path.join(DATASET_DIR,"crash")
NORMAL_DIR = os.path.join(DATASET_DIR,"normal")

CLIP_SAVE_DIR = "/content/drive/MyDrive/CCD_CLIPS_Par"

In [ ]:
os.makedirs(CLIP_SAVE_DIR,exist_ok=True)

for split in ["train","val","test"]:
    os.makedirs(os.path.join(CLIP_SAVE_DIR,split),exist_ok=True)

In [ ]:
def collect_video_paths():

    paths=[]
    labels=[]

    for f in os.listdir(CRASH_DIR):
        if f.lower().endswith((".mp4",".avi",".mov")):
            paths.append(os.path.join(CRASH_DIR,f))
            labels.append(1)

    for f in os.listdir(NORMAL_DIR):
        if f.lower().endswith((".mp4",".avi",".mov")):
            paths.append(os.path.join(NORMAL_DIR,f))
            labels.append(0)

    return np.array(paths),np.array(labels)


video_paths,labels = collect_video_paths()

print("Total videos:",len(video_paths))

Total videos: 4500


In [ ]:
X_temp,X_test,y_temp,y_test = train_test_split(
    video_paths,
    labels,
    test_size=0.15,
    stratify=labels,
    random_state=SEED
)

X_train,X_val,y_train,y_val = train_test_split(
    X_temp,
    y_temp,
    test_size=0.15,
    stratify=y_temp,
    random_state=SEED
)

print("Train:",len(X_train))
print("Val:",len(X_val))
print("Test:",len(X_test))

Train: 3251
Val: 574
Test: 675


In [ ]:
def sample_clips_from_video(video_path,label,num_clips=3):

    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if total_frames < SEQ_LEN:
        cap.release()
        return []

    positions = np.linspace(
        0,
        total_frames - SEQ_LEN,
        num_clips,
        dtype=int
    )

    frames=[]

    while True:

        ret,frame = cap.read()

        if not ret:
            break

        frame = cv2.resize(frame,(IMG_SIZE,IMG_SIZE))
        frame = frame.astype(np.float32)/255.0

        frames.append(frame)

    cap.release()

    frames = np.array(frames)

    clips=[]

    for start in positions:

        clip = frames[start:start+SEQ_LEN]

        if len(clip)==SEQ_LEN:
            clips.append((clip,label))

    return clips

In [ ]:
def process_video(args):

    video_path,label,split,index = args

    save_dir = os.path.join(CLIP_SAVE_DIR,split)

    clips = sample_clips_from_video(
        video_path,
        label,
        CLIPS_PER_VIDEO
    )

    saved = 0

    for i,(clip,label) in enumerate(clips):

        save_path = os.path.join(
            save_dir,
            f"clip_{index}_{i}.npz"
        )

        np.savez_compressed(
            save_path,
            clip=clip,
            label=label
        )

        saved += 1

    return saved

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm

def extract_and_save_clips_parallel(video_paths, labels, split):

    tasks = [(path, label, split, i) for i,(path,label) in enumerate(zip(video_paths,labels))]

    total_saved = 0

    with ThreadPoolExecutor(max_workers=4) as executor:

        results = list(
            tqdm(
                executor.map(process_video, tasks),
                total=len(tasks),
                desc=f"Processing {split}"
            )
        )

    total_saved = sum(results)

    print(split, "clips saved:", total_saved)

In [ ]:
extract_and_save_clips_parallel(X_train,y_train,"train")
extract_and_save_clips_parallel(X_val,y_val,"val")
extract_and_save_clips_parallel(X_test,y_test,"test")

Processing train: 100%|██████████| 3251/3251 [35:08<00:00,  1.54it/s]


train clips saved: 9753


Processing val: 100%|██████████| 574/574 [06:46<00:00,  1.41it/s]


val clips saved: 1722


Processing test: 100%|██████████| 675/675 [07:18<00:00,  1.54it/s]

test clips saved: 2025
